Import python packages

In [1]:
import struct
import sys
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
plt.rcParams['figure.figsize']=(6,6)
plt.rcParams['font.weight']='bold'
plt.rcParams['axes.labelweight']='bold'
plt.rcParams['lines.linewidth']=2
plt.rcParams['lines.markeredgewidth']=2
%matplotlib inline
%config InlineBackend.figure_format = "retina"

Load ThinCurr Library

In [2]:
thincurr_python_path = '/home/clair/repos/install_release'
if thincurr_python_path is not None:
    sys.path.append(os.path.join(thincurr_python_path,'python'))
from OpenFUSIONToolkit import OFT_env
from OpenFUSIONToolkit.ThinCurr import ThinCurr
from OpenFUSIONToolkit.ThinCurr.meshing import build_torus_bnorm_grid, ThinCurr_periodic_toroid
from OpenFUSIONToolkit.ThinCurr.sensor import Mirnov, save_sensors
from OpenFUSIONToolkit.ThinCurr.valen import VALENSystem 

Create 3D mode model

In [3]:
def create_circular_bnorm(filename,R0,Z0,a,n,m,npts=200):
    theta_vals = np.linspace(0.0,2*np.pi,npts,endpoint=False)
    with open(filename,'w+') as fid:
        fid.write('{0} {1}\n'.format(npts,n))
        for theta in theta_vals:
            fid.write('{0} {1} {2} {3}\n'.format(
                R0+a*np.cos(theta),
                Z0+a*np.sin(theta),
                np.cos(m*theta),
                np.sin(m*theta)
            ))
# Create n=2, m=3 mode
create_circular_bnorm('tCurr_mode.dat',1.0,0.0,0.4,2,3)

Generate Mesh and Plot

In [4]:
ntheta = 40
nphi = 80
r_grid, bnorm, nfp = build_torus_bnorm_grid('tCurr_mode.dat',ntheta,nphi,resample_type='theta',use_spline=False)
plasma_mode = ThinCurr_periodic_toroid(r_grid,nfp,ntheta,nphi)


Loading toroidal plasma mode
  filename = tCurr_mode.dat
  N        = 2
  # of pts = 200
  R0       = (1.0000E+00, -1.7087E-17)
  Mode pair sums -7.6050E-15 -1.5543E-15


Save model to HDF5 format

In [5]:
plasma_mode.write_to_file('thincurr_mode.h5')


Saving mesh: thincurr_mode.h5


Setup plasma model (mode) and wall model (torus)

In [6]:
myOFT = OFT_env(nthreads=4)
tw_mode = ThinCurr(myOFT)
tw_mode.setup_model(mesh_file='thincurr_mode.h5')
tw_mode.setup_io(basepath='plasma/')

tw_torus = ThinCurr(myOFT)
tw_torus.setup_model(mesh_file='thincurr_ex-torus.h5',xml_filename='oft_in.xml')
tw_torus.setup_io()

#----------------------------------------------
Open FUSION Toolkit Initialized
Development branch:    main
Revision id:           1a4bc3c
Parallelization Info:
  Not compiled with MPI
  # of OpenMP threads =    4
Integer Precisions    =    4   8
Float Precisions      =    4   8  10
Complex Precisions    =    4   8
LA backend            = native
#----------------------------------------------


Creating thin-wall model
  No V(t) driver coils found
  No I(t) driver coils found
  Building holes

  Setup complete:
    # of points    =         6320
    # of edges     =        18960
    # of cells     =        12640
    # of holes     =            3
    # of closures  =            2
    # of Vcoils    =            0
    # of Icoils    =            0

Creating thin-wall model
  No V(t) driver coils found
  Loading I(t) driver coils
    Masked      0 coils from sensors
  Building holes

  Loading region surface resistivity:
     1  1.2570E-05

  Setup complete:
    # of points    =         23

Create sensor file? 

In [21]:
sensors = [
    Mirnov([1.45,0.0,0.0], [0.0,0.0,1.0], 'Bz_inner'),
    Mirnov([1.55,0.0,0.0], [0.0,0.0,1.0], 'Bz_outer'),
]
save_sensors(sensors)

In [22]:
coil_current = 1.E4

In [10]:
tw_mode.compute_Lmat()
Ld = tw_mode.Lmat
Lmat_new = plasma_mode.condense_matrix(tw_mode.Lmat)
print(Lmat_new)

Building element<->element self inductance matrix
  Time = 38s          
[[ 6.87032210e-02 -4.43048484e-03 -1.60332977e-03 ... -8.91471006e-04
  -2.47098149e-02  2.75050296e-02]
 [-4.43048484e-03  6.83811638e-02 -4.26899743e-03 ... -2.97205198e-04
  -1.41374409e-02  2.75055325e-02]
 [-1.60332977e-03 -4.26899743e-03  6.78555133e-02 ... -1.33528731e-04
  -7.67646173e-03  2.75066164e-02]
 ...
 [-8.91471006e-04 -2.97205198e-04 -1.33528731e-04 ...  6.87032210e-02
   1.39721043e-02 -2.75050296e-02]
 [-2.47098149e-02 -1.41374409e-02 -7.67646173e-03 ...  1.39721043e-02
   6.54342016e+00  2.92649925e-04]
 [ 2.75050296e-02  2.75055325e-02  2.75066164e-02 ... -2.75050296e-02
   2.92649925e-04  3.14289168e+00]]


In [7]:
s = -0.5
a = 0.5
valen_model = VALENSystem(s,a,tw_torus,tw_mode)
eigs = valen_model.eigenvalues()

Building element<->element self inductance matrix
  Time = 10s          
Building element<->element mutual inductance matrix
  Time = 33s          
Building element<->element self inductance matrix
  Time = 40s          
Building resistivity matrix
Building resistivity matrix


LinAlgError: 0-dimensional array given. Array must be at least two-dimensional

Generate self inductance matrices for plasma and wall 

In [ ]:
tw_mode.compute_Lmat()
mode_Lmat = plasma_mode.condense_matrix(tw_mode.Lmat)

tw_torus.compute_Lmat()
torus_Lmat = plasma_mode.condense_matrix(tw_torus.Lmat)


Building element<->element self inductance matrix


Generate resistivity matrices

In [10]:
tw_torus.compute_Rmat()
tw_mode.compute_Rmat()

Building resistivity matrix
Building resistivity matrix


Mutual inductance between plasma, wall and coils

In [ ]:
torus_coil = tw_torus.compute_Mcoil()
mode_coil = tw_mode.compute_Mcoil()



Building coil<->element inductance matrices
  Time =  0s          
Building coil<->element inductance matrices
[]
  Time =  0s          


Mutual inductance between coils and plasma, wall

In [17]:
coil_torus = torus_coil.T
coil_mode = mode_coil.T

Mutual inductance between plasma and wall, wall and plasma

In [19]:
mode_torus = tw_mode.cross_coupling(tw_torus)
torus_mode = mode_torus.T

Building element<->element mutual inductance matrix
  Time = 32s          


Sensors

In [21]:
mode_sensor, _, _ = tw_mode.compute_Msensor('floops.loc')
torus_sensor, Msc, _ = tw_torus.compute_Msensor('floops.loc')


Loading sensor information
  Loading flux loops from file: floops.loc
    # of floops =
   
Building element->sensor inductance matrix
  Time =  0s          
Building coil->sensor inductance matrix
  No magnetic sensors or coils, skipping...

Loading sensor information
  Loading flux loops from file: floops.loc
    # of floops =
   
Building element->sensor inductance matrix
  Time =  0s          
Building coil->sensor inductance matrix
  Time =  0s          
